# VoxPrep HK: real QLoRA training

This notebook has **not been executed**. Use a CUDA GPU in Colab (primary) or Kaggle. Upload the source package and the reviewed, deduplicated dataset. Never mark synthetic drafts as human reviewed. A free GPU may not be available; no paid compute is requested. Save the final run directory before disconnecting.


In [ ]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Set this to the extracted repository path in your notebook environment.
PROJECT = Path("/content/COMP4431_GroupProject_InterviewCoach")
assert (PROJECT / "ml/config.json").is_file(), (
    "Upload and extract the repository, then set PROJECT."
)
os.chdir(PROJECT)

In [ ]:
%pip install -r ml/requirements-train.txt

## Validate human review and split integrity

Edit and review examples outside this notebook. The review record must name a real reviewer, decision, time, and current content hash. Independent second reviews are required for the stratified sample. Refreshing a changed content hash clears old approvals. The split command blocks near duplicates and template/source leakage.


In [ ]:
!python -m ml.dataset validate --input ml/data/reviewed_examples.jsonl --require-review --check-duplicates
!python -m ml.dataset split --input ml/data/reviewed_examples.jsonl --output ml/data/reviewed

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
print(torch.cuda.get_device_name(0))
print("VRAM GiB", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
!python -m ml.train --config ml/config.json --validate-only

## Train and preserve evidence

The script enforces NF4 double quantization, rank 32 / alpha 64, assistant-only loss, effective batch 32, 2,048-token limit, and best validation checkpoint retention. If you run out of GPU memory, keep batch size one and reduce memory use without changing the recorded experimental setup. Do not substitute CPU/Mac training. If the environment disconnects, resume from a retained checkpoint using `--resume-from-checkpoint PATH`.


In [ ]:
!python -m ml.train --config ml/config.json

In [ ]:
import json

run = Path("ml/runs/voxprep-qlora")
manifest = json.loads((run / "model_manifest.json").read_text())
assert manifest["status"] == "training_completed"
from IPython.display import Image, display

display(Image(filename=str(run / "loss.png")))
print("Best validation loss:", manifest["best_validation_loss"])

In [ ]:
import shutil

archive = shutil.make_archive("voxprep-training-evidence", "zip", run)
print("Download and retain:", archive)
# Colab only; Kaggle users can download the output from the Files pane.
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    pass

## Export and evaluate separately

After downloading the run, merge and convert using `python -m ml.export --run ... --llama-cpp /path/to/llama.cpp`. Keep the unfine-tuned base model for A/B comparison. Follow `docs/EVALUATION.md` for the held-out 120-prompt comparison, independent reviewer ratings, speech tests, and usability work. A successful training loss curve alone does not establish product quality.
